# 04 · Train a policy with Isaac Lab 3
**20 minutes. Objective:** perform actual short PPO updates on Newton transitions, export the saved actor, and bind the exact checkpoint/export/manifest for Notebooks05–06.

We **warm-start only the actor** from the pinned public VelStand ONNX. The critic and Adam optimizer are fresh. This is not from-scratch locomotion, full PPO resume, or a convergence demonstration. Isaac Lab owns the Newton physics; this workshop's small PPO loop is not an unmentioned RSL-RL task.

The public actor is an initialization, never an inference fallback. If your training/export fails, the route stops. Keep failed directories. All commands run in the prepared simulator image; the host kernel does not import Kit, ROS, Torch or NumPy.

In [ ]:
from pathlib import Path
import os, sys, json
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Select the host Python 3.12 kernel created by scripts/notebook_setup.sh")
# This kernel is deliberately not a ROS / Kit interpreter.
start = Path(os.environ.get("WORKSHOP_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "notebooks/workshop.py").is_file()
             and (p / "src/microduck_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository, or set WORKSHOP_ROOT")
sys.path.insert(0, str(ROOT / "notebooks"))
from policy_workshop import PolicySession, training_bundle, admitted_bundle, read_json, sha256
session = PolicySession(ROOT, work=os.environ.get("WORKSHOP_NOTEBOOK_WORK"))
VALIDATE = os.environ.get("WORKSHOP_NOTEBOOK_MODE") == "validate"
print("Host Python:", sys.version.split()[0])
print("Mode:", "safe validation (hardware SKIPPED)" if VALIDATE else "interactive")

## Predict · 4 minutes
Which of these can change during this run: actor weights, observation layout, critic weights, physical admission thresholds? Why does a nonzero optimizer step not prove walking convergence?

Read the real observation constructor and reward below. The 61 inputs include encoder/IMU estimates, previous **raw** actions and desired commands; the 14 outputs become targets through the selected action pipeline. World truth is reward/termination-only, not an extra actor input. The action-magnitude penalty is not a temporal smoothness penalty.

In [ ]:
import ast
contract_path = ROOT / "src/microduck_workshop/policy/contract.py"
contract_source = contract_path.read_text()
contract_tree = ast.parse(contract_source)
for node in contract_tree.body:
    if isinstance(node, ast.FunctionDef) and node.name in {"observation", "targets"}:
        print(ast.get_source_segment(contract_source, node))
train_source = (ROOT / "src/microduck_workshop/sim/train.py").read_text()
for node in ast.walk(ast.parse(train_source)):
    if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "reward" for t in node.targets):
        print("Actual reward source:\n" + ast.get_source_segment(train_source, node))

## Task · 3 minutes
Write your prediction in a new markdown cell: explain the tracking/uprightness/action-magnitude trade-off and identify where the desired command enters the observation. Keep the short classroom recipe at **4 updates, horizon32, seed7** for the shared comparison; changing a seed is a separate new experiment, never an overwrite.

Before opt-in: finish Notebook03 and explicitly clean up its ROS/Sim. Stop only your own model server before training on a single L40S. Ask staff to confirm GPU ownership and EULA acceptance. A free-memory reading is not an allocation. Staff must supply images containing the new API and `onnx==1.22.0`; an older `:integrated` image is not automatically compatible.

In [ ]:
RUN_TRAINING = False
ACCEPT_NVIDIA_EULA = False
GPU = os.environ.get("WORKSHOP_GPU", "0")
SIM_IMAGE = os.environ.get("WORKSHOP_SIM_IMAGE", "roscon/microduck-sim:local")
ASSETS = Path(os.environ.get("WORKSHOP_ASSET_ROOT", ROOT / ".cache/assets")).resolve()
UPDATES, HORIZON, SEED = 4, 32, 7

def prepare_training():
    session.require("03")
    active = set(session.state().get("containers", {})) & {"sim", "ros", "train", "admission", "agent"}
    if active:
        raise RuntimeError("Explicitly clean up your earlier containers first: " + str(sorted(active)))
    session.gpu_preflight(GPU, accept_eula=ACCEPT_NVIDIA_EULA)
    session.capabilities(SIM_IMAGE)  # real --help and ONNX-version checks, no source-overlay illusion
    output = session.new_output("train")
    state = session.state()
    for stage in list(state["stages"]):
        if stage >= "04":
            del state["stages"][stage]
    state["training_candidate"] = str(output / "run")  # CLI creates NEW child, not the mount root
    session.save(state)
    print("New run:", state["training_candidate"])
session.step("prepare actual warm-start training", RUN_TRAINING, prepare_training)

## Execute · 7 minutes
The next cell runs native transitions and optimizer updates, not a mock or discarded actor exercise. Its seven-minute deadline is a teaching limit, not a speed promise. Container logs are retained privately; a timeout stops only this owned job and preserves incomplete output. No retry occurs.

**Online asset prerequisite:** this native job uses Docker bridge networking for outbound HTTPS to NVIDIA's public Isaac asset server (`omniverse-content-production.s3-us-west-2.amazonaws.com`). The pinned SDK downloads its default ground USD and dependencies into this container's temporary directory. Notebook03's shared HOME warmup does not make a new container offline-ready. No ports are published and host networking is not used for training. Blocked DNS/HTTPS remains a setup failure; do not replace the ground or loosen physics.

**Expected observation, not sample output:** real `PPO_UPDATE` records and new artifact files. Finishing with exit0 alone is insufficient; inspect and validate the actual receipt next.

In [ ]:
def train_actual_actor():
    run = Path(session.state()["training_candidate"])
    if run.exists():
        raise RuntimeError("Output exists; preserve it. Explicit preparation creates a different attempt.")
    session.job("train", SIM_IMAGE,
        ["-m", "microduck_workshop.sim.train", "--asset-root", "/assets",
         "--warm-start-onnx", "/assets/policies/velstand.onnx", "--output", "/outputs/run",
         "--updates", str(UPDATES), "--horizon", str(HORIZON), "--seed", str(SEED)],
        timeout=420, memory="32g", cpus=8, network="bridge",
        options=session.sim_options(GPU, ASSETS, run.parent))
session.step("real short Newton PPO and saved-checkpoint export", RUN_TRAINING, train_actual_actor)

## Observe · 4 minutes
Read **your** `training_receipt.json`: actual update metrics, physics steps, changed actor parameters, saved optimizer steps and export parity. A healthy short run should record finite metrics and changed actor bytes; do not copy the instructor's numerical results into yours.

The checkpoint hash binds the saved training state; the actor hash binds exported inference bytes. The manifest also binds current control source and the training receipt. All three artifacts, the receipt and raw transitions must stay together. Notebook05 re-hashes them rather than trusting a filename.

In [ ]:
def record_training():
    bundle = training_bundle(ROOT, session.state()["training_candidate"])
    session.verify_bundle(SIM_IMAGE, bundle)
    receipt = read_json(Path(bundle["run"]) / "training_receipt.json")
    print(json.dumps({key: receipt[key] for key in (
        "actor_init", "critic_init", "optimizer_init", "full_ppo_resume", "converged_gait",
        "physics_steps", "actor_changed_parameters", "saved_optimizer_steps", "updates", "export_parity")}, indent=2))
    print("Your immutable handoff hashes:", json.dumps(bundle["identity"], indent=2))
    session.complete("04", bundle, artifacts=bundle["files"])
session.step("validate and persist your exact trained artifact", RUN_TRAINING, record_training)

## Debug and Cleanup · 2 minutes
- Missing new CLI flags or ONNX package: wrong prepared image. Ask staff; do not install a replacement Torch/Lab stack in this kernel.
- Training timeout/NaN/missing final receipt: incomplete run, no deployment checkpoint. Keep logs and partial artifacts; do not overwrite or stamp it successful.
- Source changed after training: reproduce a new run and admission on the final release. Never update checksums to excuse changed source.
- No GPU allocation: code-reading is useful but remains **SKIPPED**, not actual training.

State one fact your receipt proves and one it cannot prove. Clean up only your training job; artifacts remain for [05 inference](05_policy_inference.ipynb). Do not delete failed attempts or shared models.

In [ ]:
session.step("owned training cleanup; keep all artifacts", RUN_TRAINING, lambda: session.stop("train"))